In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder 
    .appName("SparkBoarchik") 
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) 
    .getOrCreate()
)

hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")




In [6]:
print(type(spark))
print("Активные Spark сессии:", spark.sparkContext._active_spark_context)

<class 'pyspark.sql.session.SparkSession'>
Активные Spark сессии: <SparkContext master=local[*] appName=SparkBoarchik>


In [23]:
import os
# ⬇️ Параметры подключения к PostgreSQL
jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")


shops_df = (
        spark.read 
        .format("jdbc") 
        .option("url", jdbc_url) 
        .option("user", db_user) 
        .option("password", db_password) 
        .option("dbtable", "public.shops") 
        .option("fetchsize", 1000) 
        .option("driver", "org.postgresql.Driver") 
        .load()
)


shop_tz_df = (
        spark.read 
        .format("jdbc") 
        .option("url", jdbc_url) 
        .option("user", db_user) 
        .option("password", db_password) 
        .option("dbtable", "public.shop_timezone") 
        .option("fetchsize", 1000) 
        .option("driver", "org.postgresql.Driver") 
        .load()
)

# # Регистрируем DataFrame-ы как временные вьюхи
shops_df.createOrReplaceTempView("shops")
shop_tz_df.createOrReplaceTempView("shop_timezone")


In [26]:
shops_df.show(2, truncate=False)
shop_tz_df.show(2, truncate=False)

+-----+---------+
|st_id|shop_name|
+-----+---------+
|842  |Lenta    |
|843  |Magnit   |
+-----+---------+
only showing top 2 rows

+-----+---------+
|plant|time_zone|
+-----+---------+
|842  |         |
|842  |RUS07    |
+-----+---------+
only showing top 2 rows



In [17]:
shops_df.printSchema()
shop_tz_df.printSchema()

root
 |-- st_id: integer (nullable = true)
 |-- shop_name: string (nullable = true)

root
 |-- plant: string (nullable = true)
 |-- time_zone: string (nullable = true)



In [128]:
st_timezone = spark.sql(
"""
WITH cte_timezone AS (
    SELECT 
         CAST(st.plant AS int) AS st_id
        ,st.time_zone
    FROM shop_timezone st
    WHERE CAST(st.plant AS int) IS NOT NULL
)
,cte_join AS (
    SELECT 
         sh.st_id
        ,sh.shop_name
        ,ct.time_zone
        ,ROW_NUMBER() OVER(PARTITION BY sh.st_id ORDER BY (TRIM(ct.time_zone)='')) AS n
    FROM shops                AS sh
    INNER JOIN cte_timezone AS ct ON ct.st_id = sh.st_id
)
SELECT 
     c.st_id
    ,c.shop_name
    ,CASE
         WHEN c.time_zone IS NULL OR TRIM(c.time_zone) = '' THEN 3
         ELSE CAST(RIGHT(TRIM(c.time_zone),2) AS int)
     END AS tz_code
FROM cte_join AS c
WHERE c.n = 1                
ORDER BY c.st_id;
"""
)

In [130]:
st_timezone.show()

+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      7|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      5|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
+-----+-----------+-------+



In [131]:
st_timezone.printSchema()

root
 |-- st_id: integer (nullable = true)
 |-- shop_name: string (nullable = true)
 |-- tz_code: integer (nullable = true)



In [132]:
spark.stop()